# Лабораторная работа №2

## Обработка признаков (часть 2)

**Цель лабораторной работы:** изучение продвинутых способов предварительной обработки данных для дальнейшего формирования моделей машинного обучения.

**Задание:**

Необходимо выполнить:

1. Масштабирование признаков не менее чем тремя способами.
2. Обработку выбросов для числовых признаков:
   - удаление выбросов;
   - замена выбросов.
3. Обработку по крайней мере одного нестандартного признака.
4. Отбор признаков:
   - метод фильтрации;
   - метод обертывания;
   - метод вложений.

В данной лабораторной работе используется искусственно сгенерированный датасет о клиентах интернет-магазина.


## 1. Импорт библиотек

На этом этапе подключаются библиотеки для работы с данными, визуализации, масштабирования признаков, обработки выбросов и отбора признаков.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)

## 2. Генерация датасета

Создается искусственный набор данных, содержащий:

- числовые признаки: возраст клиента, доход, количество покупок, средний чек;
- категориальные признаки: город и тип устройства;
- нестандартный признак: текстовый отзыв клиента;
- целевой признак: факт покупки премиум-подписки.

Дополнительно в числовые признаки искусственно добавляются выбросы.

In [ ]:
n = 300

data = pd.DataFrame({
    'Age': np.random.randint(18, 70, n),
    'Income': np.random.normal(80000, 25000, n).astype(int),
    'Purchases': np.random.poisson(6, n),
    'AvgCheck': np.random.normal(3500, 1200, n).astype(int),
    'City': np.random.choice(['Moscow', 'Kazan', 'Samara', 'Tula'], n),
    'Device': np.random.choice(['Mobile', 'Desktop', 'Tablet'], n),
    'Review': np.random.choice([
        'good fast delivery',
        'bad service slow support',
        'excellent quality and price',
        'average product normal delivery',
        'poor quality not recommend'
    ], n)
})

# Добавление выбросов
outlier_indices = np.random.choice(data.index, size=10, replace=False)
data.loc[outlier_indices[:5], 'Income'] = data.loc[outlier_indices[:5], 'Income'] * 5
data.loc[outlier_indices[5:], 'AvgCheck'] = data.loc[outlier_indices[5:], 'AvgCheck'] * 6

# Формирование целевого признака
data['Premium'] = (
    (data['Income'] > 85000) &
    (data['Purchases'] > 5) &
    (data['Review'].str.contains('good|excellent'))
).astype(int)

print('Первые строки датасета:')
display(data.head())

print('\nРазмер датасета:')
print(data.shape)

print('\nИнформация о признаках:')
display(data.info())

## 3. Первичный анализ числовых признаков

Перед обработкой выбросов необходимо посмотреть основные статистические характеристики числовых признаков. 
Особое внимание уделяется минимальным и максимальным значениям, так как они позволяют обнаружить аномальные наблюдения.

In [ ]:
numeric_features = ['Age', 'Income', 'Purchases', 'AvgCheck']

display(data[numeric_features].describe())

data[numeric_features].hist(figsize=(10, 8))
plt.suptitle('Распределение числовых признаков')
plt.show()

## 4. Масштабирование признаков тремя способами

Масштабирование используется для приведения числовых признаков к сопоставимому диапазону значений.

В работе применяются три способа:

1. **MinMaxScaler** — приводит значения к диапазону от 0 до 1.
2. **StandardScaler** — преобразует признаки так, чтобы среднее было около 0, а стандартное отклонение около 1.
3. **RobustScaler** — использует медиану и межквартильный размах, поэтому менее чувствителен к выбросам.

In [ ]:
data_scaled = data.copy()

minmax_scaler = MinMaxScaler()
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

minmax_scaled = pd.DataFrame(
    minmax_scaler.fit_transform(data[numeric_features]),
    columns=[col + '_MinMax' for col in numeric_features]
)

standard_scaled = pd.DataFrame(
    standard_scaler.fit_transform(data[numeric_features]),
    columns=[col + '_Standard' for col in numeric_features]
)

robust_scaled = pd.DataFrame(
    robust_scaler.fit_transform(data[numeric_features]),
    columns=[col + '_Robust' for col in numeric_features]
)

scaled_result = pd.concat(
    [data[numeric_features].head(), minmax_scaled.head(), standard_scaled.head(), robust_scaled.head()],
    axis=1
)

display(scaled_result)

## 5. Обработка выбросов: удаление выбросов

Для удаления выбросов используется метод межквартильного размаха IQR.

Наблюдение считается выбросом, если значение признака меньше `Q1 - 1.5 * IQR` или больше `Q3 + 1.5 * IQR`.

В данном блоке строки с выбросами удаляются из датасета.

In [ ]:
def remove_outliers_iqr(df, columns):
    result = df.copy()
    for col in columns:
        q1 = result[col].quantile(0.25)
        q3 = result[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        result = result[
            (result[col] >= lower_bound) &
            (result[col] <= upper_bound)
        ]

    return result

data_without_outliers = remove_outliers_iqr(data, ['Income', 'AvgCheck'])

print('Размер исходного датасета:', data.shape)
print('Размер датасета после удаления выбросов:', data_without_outliers.shape)

display(data_without_outliers[['Income', 'AvgCheck']].describe())

## 6. Обработка выбросов: замена выбросов

Второй способ обработки выбросов — не удалять строки, а заменить аномальные значения на граничные значения.

Такой подход позволяет сохранить количество объектов в выборке.

In [ ]:
def replace_outliers_iqr(df, columns):
    result = df.copy()
    for col in columns:
        q1 = result[col].quantile(0.25)
        q3 = result[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        result[col] = np.where(result[col] < lower_bound, lower_bound, result[col])
        result[col] = np.where(result[col] > upper_bound, upper_bound, result[col])

    return result

data_replaced_outliers = replace_outliers_iqr(data, ['Income', 'AvgCheck'])

print('Размер датасета после замены выбросов:', data_replaced_outliers.shape)

display(data_replaced_outliers[['Income', 'AvgCheck']].describe())

## 7. Обработка нестандартного признака

В качестве нестандартного признака используется текстовый признак `Review`.

Для его обработки применяется метод TF-IDF. Он преобразует текстовые отзывы в числовые признаки, которые можно использовать в моделях машинного обучения.

In [ ]:
tfidf = TfidfVectorizer(max_features=8)

review_tfidf = tfidf.fit_transform(data['Review'])

review_features = pd.DataFrame(
    review_tfidf.toarray(),
    columns=['Review_' + word for word in tfidf.get_feature_names_out()]
)

print('Признаки, полученные из текстового поля Review:')
display(review_features.head())

## 8. Подготовка итогового датасета для отбора признаков

Для отбора признаков необходимо подготовить числовую матрицу признаков.

Выполняются следующие действия:

1. Числовые признаки берутся из исходного датасета.
2. Категориальные признаки кодируются методом One-Hot Encoding.
3. Текстовый признак преобразуется методом TF-IDF.
4. Все признаки объединяются в одну таблицу.

In [ ]:
categorical_features = ['City', 'Device']

categorical_encoded = pd.get_dummies(
    data[categorical_features],
    columns=categorical_features,
    dtype=int
)

X = pd.concat(
    [
        data[numeric_features].reset_index(drop=True),
        categorical_encoded.reset_index(drop=True),
        review_features.reset_index(drop=True)
    ],
    axis=1
)

y = data['Premium']

print('Итоговая матрица признаков:')
display(X.head())

print('\nРазмер матрицы признаков:')
print(X.shape)

print('\nЦелевой признак:')
display(y.head())

## 9. Отбор признаков методом фильтрации

Методы фильтрации оценивают признаки независимо от конкретной модели машинного обучения.

В данном блоке используется метод `SelectKBest` с критерием `f_classif`, который выбирает признаки, наиболее связанные с целевым признаком.

In [ ]:
filter_selector = SelectKBest(score_func=f_classif, k=6)

X_filter = filter_selector.fit_transform(X, y)

filter_selected_features = X.columns[filter_selector.get_support()]

print('Признаки, выбранные методом фильтрации:')
print(list(filter_selected_features))

## 10. Отбор признаков методом обертывания

Методы обертывания используют модель машинного обучения и подбирают признаки с учетом качества этой модели.

В данной работе используется метод `RFE` — рекурсивное исключение признаков. 
В качестве базовой модели используется логистическая регрессия.

In [ ]:
wrapper_model = LogisticRegression(max_iter=1000)

rfe_selector = RFE(
    estimator=wrapper_model,
    n_features_to_select=6
)

X_wrapper = rfe_selector.fit_transform(X, y)

wrapper_selected_features = X.columns[rfe_selector.get_support()]

print('Признаки, выбранные методом обертывания RFE:')
print(list(wrapper_selected_features))

## 11. Отбор признаков методом вложений

Методы вложений выполняют отбор признаков во время обучения модели.

В данной работе используется `RandomForestClassifier`, который позволяет оценить важность признаков. 
На основе этих важностей выбираются наиболее значимые признаки.

In [ ]:
embedded_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

embedded_model.fit(X, y)

embedded_selector = SelectFromModel(
    embedded_model,
    prefit=True,
    threshold='median'
)

X_embedded = embedded_selector.transform(X)

embedded_selected_features = X.columns[embedded_selector.get_support()]

print('Признаки, выбранные методом вложений:')
print(list(embedded_selected_features))

feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': embedded_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

display(feature_importances.head(10))

## 12. Сравнение качества модели на разных наборах признаков

Для проверки результата обучается логистическая регрессия на разных наборах признаков:

1. На всех признаках.
2. На признаках, выбранных методом фильтрации.
3. На признаках, выбранных методом обертывания.
4. На признаках, выбранных методом вложений.

Это позволяет увидеть, как отбор признаков влияет на качество модели.

In [ ]:
def train_and_evaluate(X_data, y_data, description):
    X_train, X_test, y_train, y_test = train_test_split(
        X_data,
        y_data,
        test_size=0.3,
        random_state=42,
        stratify=y_data
    )

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    return {
        'Набор признаков': description,
        'Accuracy': accuracy
    }

results = []

results.append(train_and_evaluate(X, y, 'Все признаки'))
results.append(train_and_evaluate(pd.DataFrame(X_filter, columns=filter_selected_features), y, 'Filter methods'))
results.append(train_and_evaluate(pd.DataFrame(X_wrapper, columns=wrapper_selected_features), y, 'Wrapper methods'))
results.append(train_and_evaluate(pd.DataFrame(X_embedded, columns=embedded_selected_features), y, 'Embedded methods'))

results_df = pd.DataFrame(results)

display(results_df)

## 13. Итоговый вывод

В ходе лабораторной работы были изучены и применены продвинутые методы обработки признаков.

Были выполнены:

- масштабирование признаков тремя способами;
- удаление выбросов методом IQR;
- замена выбросов граничными значениями;
- обработка текстового признака методом TF-IDF;
- отбор признаков методом фильтрации;
- отбор признаков методом обертывания;
- отбор признаков методом вложений.

Полученный датасет был подготовлен для дальнейшего построения моделей машинного обучения.

In [ ]:
print('Лабораторная работа №2 выполнена успешно.')

print('Выполненные этапы:')
print('1. Сгенерирован датасет с числовыми, категориальными и текстовыми признаками.')
print('2. Выполнено масштабирование признаков тремя способами.')
print('3. Выполнена обработка выбросов двумя способами.')
print('4. Выполнена обработка нестандартного текстового признака.')
print('5. Выполнен отбор признаков методами filter, wrapper и embedded.')